<a href="https://colab.research.google.com/github/mahendrapd/GRST/blob/main/DDoS_RST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
data = pd.read_parquet('/content/drive/MyDrive/dataset/training.parquet')

In [4]:
cols=len(data.columns)
nfeatures = cols-1
X = data.iloc[:,0:nfeatures]
y = data.iloc[:,-1]
print(len(data),cols)

347122 55


In [5]:
targetclass='DDoS'

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [7]:
def RSTClassifier(X,y,targetclass=y[0]):
    #print(targetclass)
    dc=pd.Series(y)
    cm=dc.groupby(dc).apply(lambda px: px.index.tolist()).to_dict() #index of targetclass
    D=set(cm[targetclass])
    # D1=set(cm['Bruteforce'])
    # D2=set(cm['Botnet'])
    # D3=set(cm['Portscan'])
    # D4=set(cm['Webattack'])
    # D5=set(cm['DoS'])
    # D6=set(cm['Infiltration'])
    # D7=set(cm['DDoS'])
    print(D)
    nfeatures=len(X.columns)
    #print(cm)
    purity={}
    headers = X.keys().tolist()

    for j in range(nfeatures):
        header=headers[j]
        hdata={}
        #print("Columns:",j,header)
        dp=pd.Series(X.iloc[:,j])
        md=dp.groupby(dp).apply(lambda px: px.index.tolist()).to_dict()
        K=md.keys()
        #print(K)
        for z in K:
            common_elements=set(md[z]) & D
            val=round(len(common_elements)/len(md[z]),3)
            #print(z,len(md[z]),len(common_elements),val)
            hdata[z]=val
        purity[header]=hdata
    #print(purity['urgent'][0])
    return purity

In [8]:
pur=RSTClassifier(X_train,y_train,targetclass)

{163840, 163841, 163842, 163843, 163845, 163847, 163848, 163849, 163851, 163853, 163854, 163855, 163856, 163858, 163859, 163860, 163861, 163865, 163868, 163869, 163870, 163872, 163874, 164168, 164171, 164172, 164174, 164177, 164178, 164179, 164180, 164182, 164183, 164184, 164186, 164187, 164188, 164189, 164191, 164192, 164193, 164195, 164198, 164201, 164202, 164203, 164205, 164206, 164207, 164208, 164209, 164210, 164211, 164212, 164213, 164214, 164216, 164540, 164542, 164543, 164545, 164546, 164547, 164551, 164552, 164553, 164554, 164555, 164557, 164558, 164563, 164564, 164565, 164568, 164569, 164570, 164571, 164572, 164573, 164575, 164576, 164577, 164581, 164582, 164585, 164586, 164587, 164588, 164589, 164591, 164594, 164595, 164596, 164597, 164598, 164599, 164600, 164601, 164602, 164971, 164972, 164973, 164974, 164975, 164976, 164979, 164980, 164981, 164982, 164983, 164984, 164985, 164986, 164987, 164988, 164991, 164993, 164994, 164996, 164998, 164999, 165000, 165002, 165003, 165004,

In [9]:
#print(pur)

In [10]:
def RST_AmbigiousSample(X,pur):
    headers = X.keys().tolist()
    print(headers,len(headers),len(X))
    ambigioussamples={}
    for i in range(len(X)):
        flag=0
        sumpurity=0
        for j in range(len(headers)):
            key=X.iloc[i][headers[j]]
            if (pur[headers[j]][key] == 1 or pur[headers[j]][key] == 0):
                flag=1
                break
            else:
                sumpurity+=pur[headers[j]][key]
        if (flag == 0):
            Avgpurity=round(sumpurity/len(headers),3)
            ambigioussamples[i]=Avgpurity
    return ambigioussamples

In [11]:
AmbSample=RST_AmbigiousSample(X_train,pur)

['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle 

In [12]:
print(len(AmbSample))

27156


In [13]:
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

In [14]:
def RST_Gradient(y,sample,epoch=300):
    Theta=0.5
    LRate=0.01
    pc=0
    for i in range(epoch):
        correctcount=0
        val=0
        Th=Theta
        L=0

        for key, value in sample.items():
            if (value >= Theta and y.iloc[key] == targetclass):
                correctcount+=1
            elif (value < Theta and y.iloc[key] != targetclass):
                correctcount+=1
            else:
                val+=value
        ictotal=(len(sample)-correctcount)
        try:
          p=correctcount/len(sample)
          #L=-(Th*math.log2(p)+(1-Th)*math.log2(1-p))
          #L=-((Th/p)-((1-Th)/(1-p)))
          #L=np.tanh(L)
          #Theta=abs(round((Theta-LRate*L),3))
          #print(L)
          if (pc<=correctcount):
            Theta=abs(round(Theta-LRate*(sigmoid((val/ictotal))),3))
            pc=correctcount
          #Theta=round(Theta-LRate*(0-(val/ictotal))*(val/ictotal)*(1-(val/ictotal)),3)
        except:
          Theta=Theta
        print(i, correctcount, ictotal, len(sample), val, Theta)

    return Theta

In [15]:
Th=RST_Gradient(y_train,AmbSample,300)

0 19982 7174 27156 442.8399999999992 0.495
1 19982 7174 27156 442.8399999999992 0.49
2 19982 7174 27156 442.8399999999992 0.485
3 19982 7174 27156 442.8399999999992 0.48
4 19982 7174 27156 442.8399999999992 0.475
5 19982 7174 27156 442.8399999999992 0.47
6 19982 7174 27156 442.8399999999992 0.465
7 19982 7174 27156 442.8399999999992 0.46
8 19982 7174 27156 442.8399999999992 0.455
9 19982 7174 27156 442.8399999999992 0.45
10 19982 7174 27156 442.8399999999992 0.445
11 19982 7174 27156 442.8399999999992 0.44
12 19982 7174 27156 442.8399999999992 0.435
13 19982 7174 27156 442.8399999999992 0.43
14 19982 7174 27156 442.8399999999992 0.425
15 19982 7174 27156 442.8399999999992 0.42
16 19982 7174 27156 442.8399999999992 0.415
17 19982 7174 27156 442.8399999999992 0.41
18 19982 7174 27156 442.8399999999992 0.405
19 19982 7174 27156 442.8399999999992 0.4
20 19982 7174 27156 442.8399999999992 0.395
21 19982 7174 27156 442.8399999999992 0.39
22 19982 7174 27156 442.8399999999992 0.385
23 19982 7

In [16]:
def RST_Validation(X,y,pur,Theta):
    TP=0
    TN=0
    FP=0
    FN=0
    headers = X.keys().tolist()
    for i in range(len(X)):
        val=0
        count=0
        pval=0

        flag=0
        for j in range(len(headers)):
            #KYS=list(pur[headers[j]].keys())
            #dflag=0
            key=X.iloc[i][headers[j]]
            try:
              pval=pur[headers[j]][key]
            except KeyError:
              pval=Theta

            if ((pval == 1 or pval == 0)):
                flag=1
                break
            else:
                val+=pval
                count+=1

        if (flag == 1):
            if (pval==1 and y.iloc[i] == targetclass):
                TP+=1
            elif (pval==1 and y.iloc[i] != targetclass):
                FN+=1
            elif (pval==0 and y.iloc[i] != targetclass):
                TN+=1
            else:
                FP+=1
        else:
            Aval=round((val/count),3)
            if (Aval>=Theta and y.iloc[i] == targetclass):
                TP+=1
            elif (Aval<Theta and y.iloc[i] != targetclass):
                TN+=1
            elif (Aval>=Theta and y.iloc[i] != targetclass):
                FN+=1
            else:
                FP+=1
    try:
      Recall=round(TP/(TP+FN),4)
    except:
      Recall=0
    try:
      FPR=round(FP/(TN+FP),4)
    except:
      FPR=0
    try:
      Precision=round(TP/(TP+FP),4)
    except:
      Precision=0
    try:
      Accuracy=round((TP+TN)/(TP+TN+FP+FN),4)
    except:
      Accuracy=0
    try:
      FScore=round(2*Recall*Precision/(Recall+Precision),4)
    except:
      FScore=0
    print("TP=",TP,"\tTN=",TN,"\tFP=",FP,"\tFN=",FN)
    return Recall, FPR, Precision, FScore, Accuracy

In [17]:
Result=RST_Validation(X_train,y_train,pur,Th)
print(Result)

TP= 6049 	TN= 251855 	FP= 1682 	FN= 755
(0.889, 0.0066, 0.7824, 0.8323, 0.9906)


In [18]:
ResultTest = RST_Validation(X_test,y_test,pur,Th)
print(ResultTest)

TP= 1890 	TN= 84001 	FP= 609 	FN= 281
(0.8706, 0.0072, 0.7563, 0.8094, 0.9897)


In [19]:
def CountPatterns(pur):
  value=0
  for outer_key, outer_value in pur.items():
    acount=0
    ncount=0
    for inner_key, inner_value in outer_value.items():
      if (inner_value == 0):
        acount=acount+1
      if (inner_value == 1):
        ncount=ncount+1
    print(f"{outer_key}: {acount}, {ncount}, {len(outer_value.items())}")

In [20]:
CountPatterns(pur)

Flow Duration: 10, 0, 11
Total Fwd Packets: 13, 29, 76
Total Backward Packets: 49, 0, 50
Fwd Packets Length Total: 5, 0, 12
Bwd Packets Length Total: 38, 0, 39
Fwd Packet Length Max: 42, 0, 45
Fwd Packet Length Mean: 33, 0, 43
Fwd Packet Length Std: 40, 0, 42
Bwd Packet Length Max: 22, 0, 31
Bwd Packet Length Mean: 8, 1, 17
Bwd Packet Length Std: 10, 1, 34
Flow Bytes/s: 2, 38, 93
Flow Packets/s: 5, 4, 39
Flow IAT Mean: 4, 0, 5
Flow IAT Std: 8, 0, 9
Flow IAT Max: 10, 0, 11
Flow IAT Min: 8, 0, 9
Fwd IAT Total: 10, 0, 11
Fwd IAT Mean: 4, 0, 5
Fwd IAT Std: 8, 0, 9
Fwd IAT Max: 10, 0, 11
Fwd IAT Min: 8, 0, 9
Bwd IAT Total: 20, 0, 101
Bwd IAT Mean: 63, 0, 101
Bwd IAT Std: 43, 0, 101
Bwd IAT Max: 29, 0, 101
Bwd IAT Min: 86, 0, 101
Fwd Header Length: 0, 82, 89
Bwd Header Length: 8, 0, 9
Fwd Packets/s: 1, 8, 45
Bwd Packets/s: 17, 0, 34
Packet Length Max: 42, 0, 52
Packet Length Mean: 10, 0, 21
Packet Length Std: 13, 0, 31
Packet Length Variance: 8, 0, 13
Avg Packet Size: 11, 0, 25
Avg Fwd Segme